In [6]:
import networkx as nx
import numpy as np
import random
import itertools
from collections import Counter
import gzip
import pickle   
import os
from tqdm import tqdm
import torch
 
def get_random_instance(n, k, p):
    a = np.log(k) / np.log(n)
    r = - a / np.log(1 - p)
    v = k * n
    s = int(p * (n ** (2 * a)))
    iterations = int(r * n * np.log(n) - 1)
    parts = np.reshape(np.int64(range(v)), (n, k))
    nand_clauses = []
    for i in parts:
        nand_clauses += list(itertools.combinations(i, 2))
    edges = set()
    for _ in range(iterations):
        i, j = np.random.choice(n, 2, replace=False)
        all_potential = set(itertools.product(parts[i, :], parts[j, :]))
        all_potential -= edges
        edges |= set(random.sample(tuple(all_potential), k=min(s, len(all_potential))))
    nand_clauses += list(edges)
    ordered_edge_list = [(min(edge), max(edge)) for edge in nand_clauses]
    return Counter(ordered_edge_list).keys()
 
num_graphs = 10000
np.random.seed(123)
 
root = '/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/MWISP_BA_MIDDLE/rmp'
os.makedirs(f'{root}/raw', exist_ok=True)
ips = []
pkg_idx = 0
for problem in tqdm(range(num_graphs)):
    graph = nx.barabasi_albert_graph(n=np.random.randint(40)+200, m=4)

    max_independent_set = nx.algorithms.approximation.maximum_independent_set(graph)
    # Convert the maximum independent set to a binary indicator vector
    # 1 indicates a node is in the MIS, 0 indicates it's not
    mis_indicator = np.zeros(graph.number_of_nodes(), dtype=int)
    for node in max_independent_set:
        mis_indicator[node] = 1
    
    # Convert to torch tensor for consistency with other data
    mis_indicator_tensor = torch.from_numpy(mis_indicator).to(torch.long)
    
    # Calculate the size of the maximum independent set
    mis_size = len(max_independent_set)

    
    # Each edge (i,j) gives a constraint: x_i + x_j <= 1
    edges = list(graph.edges())
    num_nodes = graph.number_of_nodes()
    num_constraints = len(edges)
    
    # Create constraint matrix A where each row represents an edge constraint
    A = np.zeros((num_constraints, num_nodes))
    for i, (u, v) in enumerate(edges):
        A[i, u] = 1
        A[i, v] = 1
    
    # Right-hand side of constraints (all 1's for MIS)
    b = np.ones(num_constraints)
    
    # Objective coefficients (all -1's since we're minimizing the negative of the MIS)
    c = np.ones(num_nodes)  # negative because we minimize instead of maximize
    ips.append((torch.from_numpy(A).to(torch.float), torch.from_numpy(b).to(torch.float), torch.from_numpy(c).to(torch.float),graph,mis_indicator_tensor,mis_size)) 
    if len(ips) >= 1000:
            with gzip.open(f'{root}/raw/instance_{pkg_idx}.pkl.gz', "wb") as file:
                pickle.dump(ips, file)
                pkg_idx += 1
            ips = []



 


 


100%|██████████| 10000/10000 [31:47<00:00,  5.24it/s] 


In [2]:
import networkx as nx
import numpy as np
import random
import itertools
from collections import Counter
import gzip
import pickle   
import os
from tqdm import tqdm
import torch
 
def get_random_instance(n, k, p):
    a = np.log(k) / np.log(n)
    r = - a / np.log(1 - p)
    v = k * n
    s = int(p * (n ** (2 * a)))
    iterations = int(r * n * np.log(n) - 1)
    parts = np.reshape(np.int64(range(v)), (n, k))
    nand_clauses = []
    for i in parts:
        nand_clauses += list(itertools.combinations(i, 2))
    edges = set()
    for _ in range(iterations):
        i, j = np.random.choice(n, 2, replace=False)
        all_potential = set(itertools.product(parts[i, :], parts[j, :]))
        all_potential -= edges
        edges |= set(random.sample(tuple(all_potential), k=min(s, len(all_potential))))
    nand_clauses += list(edges)
    ordered_edge_list = [(min(edge), max(edge)) for edge in nand_clauses]
    return Counter(ordered_edge_list).keys()
 
num_graphs = 10000
np.random.seed(123)
 
root = '/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/MWISP_RB_SMALL/rmp'
os.makedirs(f'{root}/raw', exist_ok=True)
ips = []
pkg_idx = 0
for problem in tqdm(range(num_graphs)):
    while True:
        n, k = np.random.randint(20, 25), np.random.randint(5, 12)
        p = np.random.uniform(0.3, 1.0)
        edges = get_random_instance(n, k, p)
        G = nx.Graph()
        G.add_edges_from(edges)
        isolated = list(nx.isolates(G))
        G.remove_nodes_from(isolated)
        if 200 <= G.number_of_nodes() <= 300:
            break
    graph = G
    max_independent_set = nx.algorithms.approximation.maximum_independent_set(graph)
    # Convert the maximum independent set to a binary indicator vector
    # 1 indicates a node is in the MIS, 0 indicates it's not
    mis_indicator = np.zeros(graph.number_of_nodes(), dtype=int)
    for node in max_independent_set:
        mis_indicator[node] = 1
    
    # Convert to torch tensor for consistency with other data
    mis_indicator_tensor = torch.from_numpy(mis_indicator).to(torch.long)
    
    # Calculate the size of the maximum independent set
    mis_size = len(max_independent_set)

    
    # Each edge (i,j) gives a constraint: x_i + x_j <= 1
    edges = list(graph.edges())
    num_nodes = graph.number_of_nodes()
    num_constraints = len(edges)
    
    # Create constraint matrix A where each row represents an edge constraint
    A = np.zeros((num_constraints, num_nodes))
    for i, (u, v) in enumerate(edges):
        A[i, u] = 1
        A[i, v] = 1
    
    # Right-hand side of constraints (all 1's for MIS)
    b = np.ones(num_constraints)
    
    # Objective coefficients (all -1's since we're minimizing the negative of the MIS)
    c = np.ones(num_nodes)  # negative because we minimize instead of maximize
    ips.append((torch.from_numpy(A).to(torch.float), torch.from_numpy(b).to(torch.float), torch.from_numpy(c).to(torch.float),graph,mis_indicator_tensor,mis_size)) 
    if len(ips) >= 1000:
            with gzip.open(f'{root}/raw/instance_{pkg_idx}.pkl.gz', "wb") as file:
                pickle.dump(ips, file)
                pkg_idx += 1
            ips = []

  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [1:07:18<00:00,  2.48it/s]


In [3]:
max_independent_set = nx.algorithms.approximation.maximum_independent_set(graph)
max_independent_set

{1, 3, 4, 11, 14, 18, 21, 23, 25, 27, 30, 36, 37, 43}

In [ ]:
# BA-large
ba_large = [nx.barabasi_albert_graph(n=np.random.randint(401)+800, m=4) for _ in range(num_graphs)]
 

In [ ]:
# RB-small
rb_small = []
for _ in range(num_graphs):
    while True:
        n, k = np.random.randint(20, 25), np.random.randint(5, 12)
        p = np.random.uniform(0.3, 1.0)
        edges = get_random_instance(n, k, p)
        G = nx.Graph()
        G.add_edges_from(edges)
        isolated = list(nx.isolates(G))
        G.remove_nodes_from(isolated)
        if 200 <= G.number_of_nodes() <= 300:
            rb_small.append(G)
            break

# RB-large
rb_large = []
for _ in range(num_graphs):
    while True:
        n, k = np.random.randint(40, 55), np.random.randint(20, 25)
        p = np.random.uniform(0.3, 1.0)
        edges = get_random_instance(n, k, p)
        G = nx.Graph()
        G.add_edges_from(edges)
        isolated = list(nx.isolates(G))
        G.remove_nodes_from(isolated)
        if 800 <= G.number_of_nodes() <= 1200:
            rb_large.append(G)
            break